# Chapter 12 — Zeek·Snort 로그 구조와 웹 로그 분석 (실습)

## 실습 목표
1. 합성 Zeek conn.log·http.log 로드·결합
2. Apache combined log 정규식 파싱
3. SQLi·XSS·Path traversal 룰 적용
4. 의심 IP 추출

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy

In [ ]:
import re
import pandas as pd
import numpy as np
from io import StringIO
rng = np.random.default_rng(42)

In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

---
## Step 1. 합성 Zeek conn.log / http.log 생성·로드

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 합성 Zeek conn.log / http.log 생성·로드
# 만들 결과: make_uid(), BASE_EPOCH, uid, src, dst, ts_epoch
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def make_uid(): ...

# 2024-07-05 13:00:00 UTC. Apache 예제는 같은 시각을 +0900 으로 적으므로
# 두 소스를 그대로 합치면 9시간이 어긋납니다. Step 3 에서 UTC 로 맞춥니다.
BASE_EPOCH = ...
rows_conn, rows_http = ...
for _ in range(200):
    uid = ...
    src = ...
    dst = ...
    # Zeek 은 ts 를 에포크 초(UTC)로 기록합니다. 13장 상관 분석의 시간 축이 됩니다.
    ts_epoch = ...
    ...
    if rng.random() > 0.3: ...
        ...
conn = ...
http = ...
merged = ...
print(f'conn = ...
...
```
</details>


---
## Step 2. Apache combined 로그 파싱

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. Apache combined 로그 파싱
# 만들 결과: sample_log, PATTERN, records, web, web['status']
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
sample_log = ...
...
PATTERN = ...
    ...
records = ...
web = ...
web['status'] = ...
...
```
</details>


---
## Step 3. 통합 정규화 — 13장이 쓸 공통 스키마

Zeek 은 에포크 초, Apache 는 `+0900` 오프셋으로 시각을 적습니다. 표기가 다른 두 소스를
그대로 이어 붙이면 같은 순간이 9시간 어긋나 상관 분석이 틀어집니다. 본문 12.4절의
`normalize()` 로 **`ts`(UTC)·`src`·`dst`·`event`** 공통 열을 만듭니다.

파싱 실패는 정상 이벤트로 넘기지 않고 `errors='raise'` 로 즉시 드러냅니다.

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 통합 정규화 — 13장이 쓸 공통 스키마 — Zeek 은 에포크 초, Apache 는 `+0900` 오프셋으로 시각을 적습니다. 표기가 다른 두 소스를
# 만들 결과: normalize(), ts, utc, events, ignore_index, naive
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def normalize(df, source: ...
    ...
    if source == 'zeek':
        ts = ...
        src, dst = ...
    elif source == 'apache':
        ts = ...
                            utc = ...
        src, dst = ...
    else:
        raise ...
    return ...

events = ...
                   ignore_index = ...

...

# 정규화하지 않으면 어떻게 어긋나는지 확인합니다.
naive = ...
...

# 지원하지 않는 소스는 조용히 통과시키지 않습니다.
try:
    ...
except ValueError as exc:
    ...
```
</details>


---
## Step 4. SQLi·XSS·Path traversal 룰

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. SQLi·XSS·Path traversal 룰
# 만들 결과: classify(), web['threat']
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def classify(uri: ...
    if re.search(r'union|select|or\s+1=1', uri, re.I): ...
    if re.search(r'<script>|javascript: ...
    if '../' in uri or '/../' in uri: ...
    if re.search(r'sqlmap|nikto|nmap|curl', ua, re.I): ...
    return ...

web['threat'] = ...
...
```
</details>


In [ ]:
# 웹 로그 위협 분류 분포
import matplotlib.pyplot as plt
vc = web['threat'].value_counts()
colors = ['#1a5f7a' if k == 'BENIGN' else '#e74c3c' for k in vc.index]
fig, ax = plt.subplots(figsize=(6, 3.3))
ax.bar(vc.index.astype(str), vc.values, color=colors)
ax.set_title('웹 로그 위협 분류 분포')
ax.set_xlabel('위협 유형'); ax.set_ylabel('요청 수')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

---
## Step 5. 의심 IP 추출

In [ ]:
# ── 실습 과제 ──
# 목표: Step 5. 의심 IP 추출
# 만들 결과: suspect
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
suspect = ...
           ...
           .rename(columns = ...
           .sort_values('n_alerts', ascending = ...
...
```
</details>


---
## 정리

- Zeek 의 uid 가 다중 로그를 묶는 join 키.
- 정규식 한 줄로 Apache 로그 파싱이 가능.
- SQLi/XSS/Path traversal 키워드 룰이 1차 탐지의 기본.
- 다음 챕터에서 이런 알람들을 SIEM 으로 상관 분석한다.